# Stock Trend Forecasting with LSTM\n
## Practical Task 2: RNN/LSTM for Stock Trend Forecasting (10 Marks)

**Objective:** Build an LSTM model to predict the next day's closing price direction (up/down).

## 1. Import Libraries

In [ ]:
!pip install yfinance -q\n
\n
import yfinance as yf\n
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score\n
from sklearn.preprocessing import MinMaxScaler\n
import tensorflow as tf\n
from tensorflow.keras import layers, models\n
from tensorflow.keras.callbacks import EarlyStopping\n
\n
np.random.seed(42)\n
tf.random.set_seed(42)\n
print(f'TensorFlow: {tf.__version__}')

## 2. Download Stock Data

In [ ]:
data = yf.download('AAPL', start='2019-01-01', end='2024-12-31')\n
print(f'Shape: {data.shape}')\n
data.head()

## 3. Visualize Closing Prices

In [ ]:
plt.figure(figsize=(14, 5))\n
plt.plot(data.index, data['Close'], label='Close Price')\n
plt.title('AAPL Stock Closing Prices')\n
plt.xlabel('Date')\n
plt.ylabel('Price ($)')\n
plt.legend()\n
plt.grid(True)\n
plt.show()

## 4. Preprocess Data: Normalize and Create Sequences

In [ ]:
# Extract and normalize closing prices\n
close_prices = data['Close'].values.reshape(-1, 1)\n
scaler = MinMaxScaler(feature_range=(0, 1))\n
scaled_prices = scaler.fit_transform(close_prices)\n
\n
print(f'Original range: ${close_prices.min():.2f} - ${close_prices.max():.2f}')\n
print(f'Scaled range: {scaled_prices.min():.4f} - {scaled_prices.max():.4f}')

In [ ]:
# Create sequences and binary labels\n
def create_sequences(data, seq_length=15):\n
    X, y = [], []\n
    for i in range(seq_length, len(data)):\n
        X.append(data[i-seq_length:i, 0])\n
        # Binary: 1 if next day up, 0 if down\n
        y.append(1 if data[i, 0] > data[i-1, 0] else 0)\n
    return np.array(X), np.array(y)\n
\n
SEQUENCE_LENGTH = 15\n
X, y = create_sequences(scaled_prices, SEQUENCE_LENGTH)\n
\n
print(f'Sequences: {len(X)}')\n
print(f'X shape: {X.shape}')\n
print(f'Up days: {np.sum(y==1)} ({np.sum(y==1)/len(y)*100:.1f}%)')\n
print(f'Down days: {np.sum(y==0)} ({np.sum(y==0)/len(y)*100:.1f}%)')

## 5. Split Data (70/15/15)

In [ ]:
train_size = int(len(X) * 0.70)\n
val_size = int(len(X) * 0.15)\n
\n
X_train, y_train = X[:train_size], y[:train_size]\n
X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]\n
X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]\n
\n
# Reshape for LSTM\n
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))\n
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))\n
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))\n
\n
print(f'Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)')\n
print(f'Val: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)')\n
print(f'Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)')

## 6. Build LSTM Model

In [ ]:
model = models.Sequential([\n
    layers.LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 1)),\n
    layers.Dropout(0.3),\n
    layers.LSTM(32, return_sequences=False),\n
    layers.Dropout(0.3),\n
    layers.Dense(16, activation='relu'),\n
    layers.Dropout(0.2),\n
    layers.Dense(1, activation='sigmoid')\n
])\n
\n
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])\n
model.summary()

## 7. Train the Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)\n
\n
history = model.fit(\n
    X_train, y_train,\n
    epochs=50,\n
    batch_size=32,\n
    validation_data=(X_val, y_val),\n
    callbacks=[early_stop],\n
    verbose=1\n
)

## 8. Evaluate on Test Set

In [ ]:
y_pred_prob = model.predict(X_test)\n
y_pred = (y_pred_prob > 0.5).astype(int).flatten()\n
\n
test_accuracy = accuracy_score(y_test, y_pred)\n
print(f'\\nTest Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

## 9. Baseline Comparison

In [ ]:
baseline_up = accuracy_score(y_test, np.ones_like(y_test))\n
baseline_down = accuracy_score(y_test, np.zeros_like(y_test))\n
baseline_random = accuracy_score(y_test, np.random.randint(0, 2, len(y_test)))\n
\n
print(f'Always UP: {baseline_up:.4f} ({baseline_up*100:.2f}%)')\n
print(f'Always DOWN: {baseline_down:.4f} ({baseline_down*100:.2f}%)')\n
print(f'Random: {baseline_random:.4f} ({baseline_random*100:.2f}%)')\n
print(f'LSTM Model: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')\n
\n
improvement = (test_accuracy - max(baseline_up, baseline_down)) * 100\n
print(f'\\nImprovement: {improvement:.2f}%')

## 10. Plot Training History

In [ ]:
plt.figure(figsize=(14, 5))\n
\n
plt.subplot(1, 2, 1)\n
plt.plot(history.history['accuracy'], label='Train')\n
plt.plot(history.history['val_accuracy'], label='Val')\n
plt.title('Accuracy')\n
plt.xlabel('Epoch')\n
plt.ylabel('Accuracy')\n
plt.legend()\n
plt.grid(True)\n
\n
plt.subplot(1, 2, 2)\n
plt.plot(history.history['loss'], label='Train')\n
plt.plot(history.history['val_loss'], label='Val')\n
plt.title('Loss')\n
plt.xlabel('Epoch')\n
plt.ylabel('Loss')\n
plt.legend()\n
plt.grid(True)\n
\n
plt.tight_layout()\n
plt.show()

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)\n
\n
plt.figure(figsize=(8, 6))\n
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',\n
            xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'])\n
plt.title('Confusion Matrix')\n
plt.ylabel('True')\n
plt.xlabel('Predicted')\n
plt.show()\n
\n
print(cm)

## 12. Classification Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Down', 'Up']))

## 13. Visualize Predictions

In [ ]:
n = 100\n
plt.figure(figsize=(14, 6))\n
plt.plot(range(n), y_test[:n], 'go-', label='Actual', alpha=0.7, markersize=4)\n
plt.plot(range(n), y_pred[:n], 'rx-', label='Predicted', alpha=0.7, markersize=4)\n
plt.title(f'Actual vs Predicted (First {n} samples)')\n
plt.xlabel('Sample')\n
plt.ylabel('Direction (0=Down, 1=Up)')\n
plt.legend()\n
plt.grid(True)\n
plt.show()

## 14. Analysis and Discussion\n
\n
### Sequence Length Choice (15 days):\n
- Captures ~3 weeks of trading patterns\n
- Balance between short-term trends and noise\n
- Too short: May miss patterns\n
- Too long: May include irrelevant data\n
\n
### Preprocessing Choices:\n
- **MinMax Scaling**: Normalizes to [0,1]\n
- **Binary Labels**: Simplifies to direction prediction\n
- **70/15/15 Split**: Standard with validation\n
\n
### Model Architecture:\n
- **Two LSTM layers**: Captures temporal dependencies\n
- **Dropout (0.3, 0.3, 0.2)**: Prevents overfitting\n
- **Sigmoid output**: Binary classification\n
\n
### Is LSTM Better Than Baseline?\n
Check the improvement percentage above. If >5%, model learned patterns. If <5%, marginal improvement.\n
\n
### Why Stock Prediction is Challenging:\n
1. **Market Efficiency**: Prices reflect available info\n
2. **Random Walk Theory**: Prices follow random patterns\n
3. **External Factors**: News, sentiment not in price data\n
4. **Non-Stationarity**: Market dynamics change\n
5. **Noise vs Signal**: High noise in daily prices\n
6. **Limited Features**: Only closing prices used\n
7. **Overfitting Risk**: Historical patterns may not repeat\n
\n
### Potential Improvements:\n
- Add features: Volume, moving averages, RSI, MACD\n
- Sentiment analysis from news\n
- Different sequence lengths\n
- Attention mechanisms\n
- Ensemble models\n
- Transformer architecture\n
\n
### ⚠️ DISCLAIMER:\n
This model is for EDUCATIONAL purposes only. DO NOT use for actual trading!

## 15. Save Model (Optional)

In [ ]:
model.save('stock_lstm_model.h5')\n
print('Model saved!')